# Track A — Fase 3: Generate folds_v2.csv + class_weights_v2.npy
**BDC Satria Data 2026 — Klasifikasi Citra Sampah**

Notebook ini menghasilkan artefak v2 final:
- `folds_v2.csv` — dataset train bersih (drop/relabel dari cleaning_log)
- `class_weights_v2.npy` — bobot kelas dari distribusi baru
- `cleaning_summary.json` — ringkasan untuk report

> **Prasyarat:** `cleaning_log.csv` sudah diisi dan tervalidasi (Fase 1 & 2 selesai)

---

In [ ]:
# ─── Cell 1: Setup ────────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import subprocess, sys
result = subprocess.run(
    ['git', 'clone', 'https://github.com/agaggigit/satria-data-bdcugm02.git', '/content/repo'],
    capture_output=True, text=True
)
print(result.stdout or result.stderr)
sys.path.insert(0, '/content/repo')
print('✅ Setup selesai')

In [ ]:
# ─── Cell 2: Konfigurasi Path ─────────────────────────────────────────────────
# ⚠️ SESUAIKAN path di bawah!

DRIVE_BASE       = '/content/drive/MyDrive/BDC2026 apace'
FOLDS_CSV        = f'{DRIVE_BASE}/output_trackA/folds.csv'          # v1 (asli)
CLEANING_LOG     = f'{DRIVE_BASE}/output_trackA/cleaning_log.csv'   # hasil review
OOF_V1_PATH      = f'{DRIVE_BASE}/output_trackB/oof.npy'            # opsional (untuk verifikasi)
OUTPUT_DIR       = f'{DRIVE_BASE}/output_trackA'

import os
for name, path in [
    ('folds.csv', FOLDS_CSV),
    ('cleaning_log.csv', CLEANING_LOG),
]:
    status = '✅' if os.path.exists(path) else '❌ TIDAK ADA'
    print(f'  {status} {name}: {path}')

In [ ]:
# ─── Cell 3: Validasi Cleaning Log Dulu ──────────────────────────────────────
from track_a.src.contact_sheet import validate_cleaning_log

log = validate_cleaning_log(CLEANING_LOG)
print('\n→ Jika ada warning, perbaiki cleaning_log.csv sebelum lanjut!')

In [ ]:
# ─── Cell 4: Generate Artefak V2 ─────────────────────────────────────────────
from track_a.src.generate_v2 import run_generate_v2

results = run_generate_v2(
    folds_csv         = FOLDS_CSV,
    cleaning_log_path = CLEANING_LOG,
    output_dir        = OUTPUT_DIR,
    oof_v1_path       = OOF_V1_PATH if os.path.exists(OOF_V1_PATH) else None,
    cap_pct           = 3.0,   # hard cap: max 3% dari train yang boleh di-drop
)

In [ ]:
# ─── Cell 5: Verifikasi Final ─────────────────────────────────────────────────
import pandas as pd
import numpy as np

folds_v2   = results['folds_v2']
weights_v2 = results['weights_v2']

print('=' * 55)
print('VERIFIKASI FINAL folds_v2.csv')
print('=' * 55)
print(f'Shape    : {folds_v2.shape}')
print(f'Kolom    : {list(folds_v2.columns)}')
print(f'Folds    : {sorted(folds_v2["fold"].unique())}')

print('\nDistribusi label:')
print(folds_v2['label'].value_counts().sort_index())

print('\nDistribusi per fold:')
print(folds_v2.groupby(['fold', 'label']).size().unstack(fill_value=0))

print(f'\nclass_weights_v2: {weights_v2.round(4).tolist()}')

# Pastikan tidak ada overlap antar fold
folds_list = sorted(folds_v2['fold'].unique())
all_ok = True
for i in range(len(folds_list)):
    for j in range(i+1, len(folds_list)):
        fps_i = set(folds_v2[folds_v2['fold'] == folds_list[i]]['filepath'])
        fps_j = set(folds_v2[folds_v2['fold'] == folds_list[j]]['filepath'])
        if fps_i & fps_j:
            print(f'❌ LEAKAGE! Fold {folds_list[i]} & {folds_list[j]}')
            all_ok = False
if all_ok:
    print('\n✅ No-overlap antar fold: KONFIRMASI')

In [ ]:
# ─── Cell 6: Cek Track B Bisa Load ───────────────────────────────────────────
# Simulasikan apa yang akan dilakukan Track B saat load folds_v2

FOLDS_V2_PATH    = f'{OUTPUT_DIR}/folds_v2.csv'
WEIGHTS_V2_PATH  = f'{OUTPUT_DIR}/class_weights_v2.npy'

# Load ulang dari file (bukan dari memory)
folds_v2_check   = pd.read_csv(FOLDS_V2_PATH)
weights_v2_check = np.load(WEIGHTS_V2_PATH)

print(f'folds_v2.csv loaded    : {len(folds_v2_check):,} baris ✅')
print(f'class_weights_v2.npy  : {weights_v2_check.round(4).tolist()} ✅')

# Simulasikan make_fold_loaders (tanpa dataset asli — hanya cek DataFrame)
df_train = folds_v2_check[folds_v2_check['fold'] != 0]
df_val   = folds_v2_check[folds_v2_check['fold'] == 0]
print(f'\nFold 0 simulation:')
print(f'  Train: {len(df_train):,} | Val: {len(df_val):,}')
print(f'  Label train: {dict(df_train["label"].value_counts().sort_index())}')
print(f'  Label val  : {dict(df_val["label"].value_counts().sort_index())}')
print('\n✅ Track B siap load folds_v2.csv!')

In [ ]:
# ─── Cell 7: Tampilkan Ringkasan untuk Laporan ────────────────────────────────
import json

summary_path = f'{OUTPUT_DIR}/cleaning_summary.json'
with open(summary_path) as f:
    summary = json.load(f)

print('=' * 55)
print('RINGKASAN CLEANING — Untuk Laporan Panitia')
print('=' * 55)
print(f"Total sebelum cleaning : {summary['total_before']:,}")
print(f"Total sesudah cleaning : {summary['total_after']:,}")
print(f"Dropped                : {summary['n_dropped']:,} ({summary['drop_pct']}%)")
print(f"Relabeled              : {summary['n_relabeled']:,}")
print(f"Kept (no change)       : {summary['n_kept']:,}")
print(f"\nClass distribution v2  : {summary['class_distribution']}")
print(f"Class weights v2       : {[round(w,4) for w in summary['class_weights_v2']]}")

In [ ]:
# ─── Cell 8: Umumkan GATE G2 ─────────────────────────────────────────────────
print('=' * 60)
print('🚦 GATE G2 — Pesan untuk dikirim ke grup:')
print('=' * 60)
print()
print('"GATE G2 HIJAU ✅')
print()
print(f'folds_v2.csv + class_weights_v2.npy sudah tersedia di Drive.')
print(f'Path: [DRIVE_BASE]/output_trackA/')
print()
print(f'Ringkasan cleaning:')
print(f'  - Dropped   : {summary["n_dropped"]} ({summary["drop_pct"]}%)')
print(f'  - Relabeled : {summary["n_relabeled"]}')
print(f'  - Total v2  : {summary["total_after"]:,} sampel')
print()
print('Track B: silakan retrain dengan folds_v2.csv (update path di config.py).')
print('Track C: update path jika ada perubahan pada konvensi split."')

---

## ✅ Fase 3 Selesai! GATE G2 Terbuka.

Artefak yang dihasilkan di Drive:
- `folds_v2.csv` — train bersih (skema identik dengan folds.csv)
- `class_weights_v2.npy` — bobot kelas untuk CrossEntropyLoss
- `cleaning_summary.json` — metodologi + angka untuk verifikasi panitia

**Track B:** Update `config.py` → `folds_csv = '...folds_v2.csv'` + `class_weights_path = '...class_weights_v2.npy'`

**Track A:** Tulis bagian *Data Cleaning* di laporan berdasarkan `cleaning_summary.json`.
